# 02 · Clip raster, checked against rasterio

## What this checks

**Clip raster** cuts a raster down to an area. It keeps the cells inside a boundary, called the cutline, and drops the
rest. This check covers release 1.3.0.

The widget can decide which cells count as inside in different ways:

- **cell centre inside:** a cell is kept when its centre is inside the cutline;
- **all touched:** a cell is kept when the cutline touches it at all;
- **all touched with a margin:** as above, plus a band of extra cells around the edge.

Which cells are kept changes any total taken from the clip, such as a population count. So it is worth checking.

## How it is checked

We repeat each clip with **rasterio**, Python's link to GDAL. GDAL is the raster library most GIS software uses. Its
`rasterio.mask` tool does the same kind of clip. Nothing here uses Fieldwork's code. Each clip is rebuilt from the
GeoTIFF, the cutline and the case's settings.

The data are made up for the test. They are a grid of 24 by 18 cells in longitude and latitude, with values that rise
in a fixed pattern. So a wrong value can be traced to its cell. The cutline is concave, and its corners sit **exactly
on the corners of cells**. That is where programs differ most in how they treat edges, so the data aim at it on
purpose.

Run notebook 00 first.

## What should happen

For each of the three cases:

1. The same window is cut from the grid.
2. The same cells are kept.
3. The kept cells hold the same values.

These criteria were fixed before the check was run.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
#
# Only what this notebook imports, and that is not tidiness. Measured in this kernel on
# 2026-10-11: with pyproj and rasterio in the same Pyodide kernel, rasterio.open crashes the
# runtime -- RuntimeError: null function, inside the GDAL wasm module -- and it kills the
# kernel rather than raising, so no cell records an error. Five of five trials crashed; the
# set below passed every trial. The mechanism is not explained here; see 00-kernel-check for
# the full map and what it does and does not establish.
NEEDED = ["numpy", "rasterio", "scipy", "shapely"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    print("Falling back to piplite for anything still missing.")
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_02_clip import run

report = run(Path.cwd() / "fixtures" / "clip-all-touched-001.json")
print(json.dumps(report["environment"], indent=2))
for case in report["cases"]:
    o = case["options"]
    print(f"\n{o['method']} margin {o.get('marginPixels', 0)}")
    print(f"  reference {case['referenceIncluded']:4d} cells   fieldwork {case['fieldworkIncluded']:4d} cells")
    print(f"  included sets agree: {case['maskAgrees']}   window agrees: {case['windowAgrees']}")
    if case["marginModelledAs"]:
        print(f"  margin modelled as: {case['marginModelledAs']}")

## The result on record

First recorded on 11 October 2026, against release 1.0.0. Rerun on 26 November 2026, against release 1.3.0:
**Partly agrees**. Each case is recorded on its own, not as one verdict.

- **Cell centre inside: agrees exactly.** The same window, the same 132 cells, and the same kept values.
- **All touched: 2 of 146 cells differ.** GDAL keeps both of them and Fieldwork does not. Both sit beside a cutline
  corner that lies exactly on a cell edge.
- **All touched with a one-pixel margin: could not be compared.** The windows differ, and 61 cells differ. But the
  two sides do not define the margin the same way (below).

Running the cell above repeats the check on the same data.

## Where the two disagree

A count alone does not say whether a difference follows a pattern. The cell below asks two questions of every cell
that differs. Which side kept it? Does it sit beside a cutline corner lying exactly on a cell edge?

In [ ]:
for case in report["cases"]:
    rows = case["differingCells"]
    if not rows:
        continue
    o = case["options"]
    reference_only = sum(1 for r in rows if r["reference"] and not r["fieldwork"])
    fieldwork_only = sum(1 for r in rows if r["fieldwork"] and not r["reference"])
    aligned = sum(1 for r in rows if r["besidePixelAlignedCutline"])
    print(f"{o['method']} margin {o.get('marginPixels', 0)}: {case['differingCellCount']} differing")
    print(f"  reference only {reference_only}, fieldwork only {fieldwork_only}")
    print(f"  beside a pixel-aligned cutline coordinate: {aligned} of {len(rows)} reported")

## How the all-touched case got to 2 cells

This check first found 35 cells that differed on all touched. Fieldwork kept cells that GDAL dropped, and never the
other way round. Every difference sat beside a cutline corner lying exactly on a cell edge.

The cause was Fieldwork's own rule. It measured the distance from the cutline to each cell's rectangle, edges
included. So a cutline running exactly along the border between two cells touched both. GDAL's all-touched rule
gives such an edge to one side only. A tiny pad on the window, a billionth of a pixel, also widened the cut by one
more cell.

That mattered because the difference ran one way. Fieldwork's all-touched clip kept more edge cells than a GDAL clip.
For a raster of counts, such as population, that would inflate any total taken from the clip. A figure from one rule
could not be compared with a figure from the other, unless the rule was stated.

Fieldwork then adopted GDAL's rule: a cell counts as touched only where the cutline covers some of its area. This
check confirmed the change. The 2 cells that remain are what is left after it.

## The margin case cannot be decided here

rasterio has no setting for a margin. So the check models a one-pixel margin as a dilation: every kept cell also keeps
its eight neighbours, a 3 by 3 square. The check labels this as a model.

Fieldwork defines its margin as a distance from each cell's rectangle. No grid rule matches that distance exactly. So
the 61 cells measure the gap to the model, not a proven disagreement. This comparison is unresolved; it does not show
a defect.

## What this does not show

- That either rule is the right one for a given analysis. GDAL is the reference here, not an authority on what a
  public-health question needs.
- Anything beyond this one case: one made-up grid, one cutline, longitude and latitude only, one band, and decimal
  values.
- Anything about reprojection, resampling, rasters with several bands, compression, or large files.